In [1]:
import pandas as pd 
import numpy as np 
import parquet

import geopandas as gpd
import requests

import seaborn as sns 
import matplotlib.pyplot as plt 

In [19]:
df = pd.read_parquet('bikeshare_clean.parquet')
df.head()

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
0,43682415,248.0,7719.0,2026-01-01 00:00:26,Wolseley St / Augusta Ave,7037.0,2026-01-01 00:04:34,Bathurst St / Dundas St W,122.0,Member,ICONIC
1,43682417,295.0,8001.0,2026-01-01 00:02:05,Union Station (North) - SMART,7076.0,2026-01-01 00:07:00,York St / Queens Quay W,4900.0,Member,ICONIC
2,43682418,2701.0,7172.0,2026-01-01 00:04:15,Strachan Ave / Princes' Blvd,7100.0,2026-01-01 00:49:16,Dundas St E / Regent Park Blvd,1833.0,Member,ICONIC
3,43682419,581.0,7259.0,2026-01-01 00:04:29,Lower Spadina Ave / Lake Shore Blvd W,7076.0,2026-01-01 00:14:10,York St / Queens Quay W,5078.0,Casual,ICONIC
4,43682420,600.0,7259.0,2026-01-01 00:04:53,Lower Spadina Ave / Lake Shore Blvd W,7076.0,2026-01-01 00:14:53,York St / Queens Quay W,3951.0,Member,ICONIC


In [161]:
df = df[df.start_time.dt.year!=2000]

In [164]:
df.info()

<class 'pandas.DataFrame'>
Index: 44599780 entries, 0 to 44599780
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int64         
 1   trip_duration       float64       
 2   start_station_id    float32       
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_station_id      float32       
 6   end_time            datetime64[us]
 7   end_station_name    category      
 8   bike_id             float32       
 9   user_type           category      
 10  bike_model          category      
dtypes: category(4), datetime64[us](2), float32(3), float64(1), int64(1)
memory usage: 2.4 GB


In [ ]:

# 3. Individual geocoding for leftover gaps — a handful of stations (Rogers Stadium, St. Andrew Subway Station, 
# specific street intersections, etc.) 
# still weren't in any snapshot, so we geocoded their names one at a time via Nominatim/OpenStreetMap. A few names turned out not to be real
# geocodable places at all (PBSC-OPS, Wolfpack, Make Invisible, NycLab) — those stayed unmatched.

# 4. Name-based fallback + fuzzy matching — some years (chiefly 2017) never recorded a station_id in the trip data at all, only a name. 
# For those, we matched on start_station_name against the combined stations table instead of ID, with a normalization/fuzzy-matching pass 
# (the one we're extending right now) to catch spelling/punctuation variants across years without misassigning a genuinely different street.


# Stations

## Live GBFS feed 

In [ ]:
# fetch station_information from Toronto Bike Share's public API (active right now)
url = "https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
stations = pd.json_normalize(requests.get(url).json()["data"]["stations"])
stations = stations[["station_id", "name", "lat", "lon"]].copy()
stations['station_id'] = pd.to_numeric(stations['station_id'], errors='coerce')

In [166]:
# transform stations to gdf

stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['lon'], stations['lat']),
    crs='EPSG:4326'
)


In [5]:
# requests.get(url).json()

## Historical snapshots

In [ ]:
# url = "https://web.archive.org/cdx/search/cdx?url=tor.publicbikesystem.net/ube/gbfs/v1/en/station_information&output=json"

# Historical snapshots via the Wayback Machine — we queried web.archive.org's CDX API to find every time that same 
# live endpoint had been archived, found 15 captures spanning 2017–2026, and fetched each one's JSON the same way as t
# he live feed. Unioned all 15 + the live feed, deduplicated by station_id (keeping the most recent name/coordinates per ID) — 
# this is the stations/all_stations table everything else builds on.

In [6]:
# query web.archive.org's CDX API to find every to find archived station_information (found 15 captures spanning 2017–2026)
timestamps = [
    '20170726114519','20190819094031','20191021144817','20210507022246',
    '20211027025026','20220307133912','20220706122732','20230326051048',
    '20240413054106','20241004201519','20250425175804','20250817130624',
    '20251010082450','20260305164528','20260310030434'
]

hist_frames = []
for ts in timestamps:
    archived_url = f"https://web.archive.org/web/{ts}if_/https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information"
    try:
        resp = requests.get(archived_url, timeout=30)
        data = resp.json()["data"]["stations"]
        frame = pd.json_normalize(data)[["station_id", "name", "lat", "lon"]]
        hist_frames.append(frame)
        print(ts, "->", len(frame), "stations")
    except Exception as e:
        print(ts, "-> FAILED:", e)


20170726114519 -> 200 stations
20190819094031 -> 462 stations
20191021144817 -> 456 stations
20210507022246 -> 608 stations
20211027025026 -> 613 stations
20220307133912 -> 617 stations
20220706122732 -> 628 stations
20230326051048 -> 655 stations
20240413054106 -> 794 stations
20241004201519 -> 852 stations
20250425175804 -> 877 stations
20250817130624 -> 953 stations
20251010082450 -> 989 stations
20260305164528 -> 1015 stations
20260310030434 -> 1015 stations


In [ ]:
all_stations = pd.concat(hist_frames + [stations], ignore_index=True)
all_stations["station_id"] = pd.to_numeric(all_stations["station_id"], errors="coerce")
# all_stations = all_stations.dropna(subset=["station_id", "lat", "lon"]).drop_duplicates(subset="station_id", keep="last")
# all_stations.shape
all_stations = all_stations.dropna(subset=["station_id", "name"])



In [172]:
all_stations.head()

,station_id,name,lat,lon
1352,7254,Borden St / Bloor St W - SMART,43.665461,-79.408480
1482,7402,Wellington St W / Bathurst St,43.642624,-79.402459
1533,7456,Belmont St / Davenport Rd - SMART,43.674381,-79.392489
1546,7470,York St / Lake Shore Blvd W,43.641960,-79.380806
1557,7481,Westmount Ave / St Clair Ave W - SMART,43.678510,-79.442150


In [74]:
import re 
def normalize_station_name(name):
    if pd.isna(name):
        return name
    s = str(name).strip()
    s = re.sub(r"[’']", "", s)      
    s = re.sub(r"\.", "", s)             
    s = re.sub(r"\s+", " ", s)           
    return s

In [76]:
all_stations['name'] = [normalize_station_name(name) for name in all_stations.name]

In [17]:
# all_stations.to_parquet('stations.parquet', index=False)

In [16]:
# all_stations_gdf = gpd.GeoDataFrame(
#     all_stations,
#     geometry=gpd.points_from_xy(all_stations["lon"], all_stations["lat"]),
#     crs="EPSG:4326"
# )


## From bikeshare data sets 

In [78]:
bs_stations = df[['start_station_id', 'start_station_name']].dropna().drop_duplicates().reset_index(drop=True)

In [79]:
bs_stations['start_station_id'] = bs_stations.start_station_id.astype('int32')

In [112]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44599781 entries, 0 to 44599780
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   trip_id             int64         
 1   trip_duration       float64       
 2   start_station_id    float32       
 3   start_time          datetime64[us]
 4   start_station_name  category      
 5   end_station_id      float32       
 6   end_time            datetime64[us]
 7   end_station_name    category      
 8   bike_id             float32       
 9   user_type           category      
 10  bike_model          category      
dtypes: category(4), datetime64[us](2), float32(3), float64(1), int64(1)
memory usage: 2.1 GB


In [122]:
df.loc[(df.start_time.dt.year==2026) & (df.start_station_id==7000.0), ['start_station_name']].drop_duplicates()

,start_station_name
145,Fort York Blvd / Capreol Ct


In [80]:
dup_names = bs_stations[bs_stations.duplicated(subset='start_station_id', keep=False)].sort_values('start_station_id')
dup_names

,start_station_id,start_station_name
1531,7000,Fort York Blvd / Capreol Crt
100,7000,Fort York Blvd / Capreol Ct
1376,7001,Lower Jarvis St / The Esplanade
62,7001,Wellesley Station Green P
144,7002,St. George St / Bloor St W
...,...,...
1186,8156,Scarborough Golf Club Rd / Lawrence Ave E
380,8171,Grafton Ave / Roncesvalles Ave
1185,8171,Grafton Ave / Roncesvalles Ave
1192,8181,Cherry St / Commissioners St


In [ ]:
# let's just normalize with punctuation and have all variations of name for current id in station dictionary. 
# after asigning id for df_year, I can change the name according the last availible version 

In [81]:
bs_stations.head()

,start_station_id,start_station_name
0,7719,Wolseley St / Augusta Ave
1,8001,Union Station (North) - SMART
2,7172,Strachan Ave / Princes' Blvd
3,7259,Lower Spadina Ave / Lake Shore Blvd W
4,7713,Temperance St / Bay St


In [82]:
bs_stations['start_station_name'] = [normalize_station_name(name) for name in bs_stations.start_station_name]

In [83]:
bs_stations.shape

(1559, 2)

In [84]:
bs_stations = bs_stations[~bs_stations.duplicated()]

In [85]:
(all_stations[['station_id', 'name']]
    .merge(bs_stations, 
           how='outer', 
           left_on  = ['station_id', 'name'], 
           right_on=['start_station_id', 'start_station_name'],
           indicator = True
           )
    .drop_duplicates() 
    .query("_merge.isin(['left_only', 'right_only'])")  
    )

,station_id,name,start_station_id,start_station_name,_merge
0,NaN,NaN,7000.0,Fort York Blvd / Capreol Crt,right_only
16,7000.0,Ft York / Capreol Crt,NaN,NaN,left_only
31,7001.0,Wellesley Station Green P (EBS),NaN,NaN,left_only
89,7006.0,Bay St / College St,NaN,NaN,left_only
120,NaN,NaN,7007.0,College St W / Huron St,right_only
...,...,...,...,...,...
11897,8266.0,Eglinton GO – Bellamy Park,NaN,NaN,left_only
11902,8272.0,Kipling Station,NaN,NaN,left_only
11903,NaN,NaN,8273.0,Exhibition Place _ Coca Cola Coliseum,right_only
11904,8273.0,Exhibition Place – Coca Cola Coliseum,NaN,NaN,left_only


In [100]:
bs_stations = bs_stations.rename(columns = {'start_station_id':'station_id', 'start_station_name': 'name'})

all_stations_pairs = pd.concat([all_stations[['station_id', 'name']], bs_stations]).dropna().drop_duplicates()
all_stations_pairs.shape

(1699, 2)

In [101]:
all_stations_pairs.sort_values('station_id')

,station_id,name
0,7000,Ft York / Capreol Crt
1531,7000,Fort York Blvd / Capreol Crt
200,7000,Fort York Blvd / Capreol Ct
1,7001,Lower Jarvis St / The Esplanade
2957,7001,Wellesley Station Green P (EBS)
...,...,...
11801,8271,RC Harris Treatment Plant
11802,8272,Kipling Station
1096,8273,Exhibition Place _ Coca Cola Coliseum
11803,8273,Exhibition Place – Coca Cola Coliseum


In [104]:
stations_vocab = (all_stations_pairs
    .drop_duplicates(subset=['station_id', 'name'], keep='last')
    .sort_values('station_id')
    .reset_index(drop=True)  
)

In [107]:
stations_vocab.to_parquet('stations_vocab.parquet', index=False)

In [131]:
dup_names

,start_station_id,start_station_name
1531,7000,Fort York Blvd / Capreol Crt
100,7000,Fort York Blvd / Capreol Ct
1376,7001,Lower Jarvis St / The Esplanade
62,7001,Wellesley Station Green P
144,7002,St. George St / Bloor St W
...,...,...
1186,8156,Scarborough Golf Club Rd / Lawrence Ave E
380,8171,Grafton Ave / Roncesvalles Ave
1185,8171,Grafton Ave / Roncesvalles Ave
1192,8181,Cherry St / Commissioners St


In [134]:
df.loc[(df.start_time.dt.year==2026) & (df.start_station_id==8181.0), ['start_station_name']].drop_duplicates()

,start_station_name
2203,Commissioners St / Cherry St


In [135]:
dedup_bs_stations = bs_stations.drop_duplicates('station_id', keep='first')
dedup_bs_stations[dedup_bs_stations['station_id']==8181]

,station_id,name
523,8181,Commissioners St / Cherry St


In [136]:
dedup_bs_stations.to_parquet('stations_actual_names.parquet')

In [176]:
all_with_neigh = gpd.sjoin(all_stations_gdf, neigh, how="left", predicate="within")
all_with_neigh = all_with_neigh[["station_id", "name", "AREA_NAME"]].rename(columns={"AREA_NAME": "neighbourhood"})

station_to_neigh_full = all_with_neigh.set_index("station_id")["neighbourhood"]
df["start_neighbourhood"] = df["start_station_id"].map(station_to_neigh_full)

In [177]:
print("trips still unmatched overall:", df["start_neighbourhood"].isna().sum(), "/", len(df))


trips still unmatched overall: 1398634 / 44599780


In [ ]:
# big improvement, unmatched trips dropped from 4.54M (10.2%) to 1.4M (3.1%) across the whole dataset. 

In [178]:
by_year = df.assign(year=df['start_time'].dt.year)
summary = by_year.groupby('year').agg(
    total_trips=('start_neighbourhood', 'size'),
    unmatched=('start_neighbourhood', lambda s: s.isna().sum())
)
summary['pct_unmatched'] = (summary['unmatched'] / summary['total_trips'] * 100).round(2)
print(summary)


      total_trips  unmatched  pct_unmatched
year                                       
2016       367961     367961         100.00
2017      1492368    1026892          68.81
2018      1922955          1           0.00
2019      2439473         77           0.00
2020      2909403        793           0.03
2021      3569606        334           0.01
2022      4615878        626           0.01
2023      5708745          0           0.00
2024      6944552          1           0.00
2025      7807215        551           0.01
2026      6821624       1398           0.02


In [111]:
mask_2026 = (df['start_time'].dt.year == 2026) & df['start_neighbourhood'].isna()
sub = df.loc[mask_2026]
print(sub['start_station_id'].value_counts())
print(sub['start_station_name'].value_counts())
print(sub['start_time'].min(), sub['start_time'].max())


start_station_id
8222.0    4
Name: count, dtype: int64
start_station_name
NycLab                                      4
 George St S / The Esplanade                0
 Grafton Ave / Roncesvalles Ave             0
 Silverstone Dr / Stevenson Rd              0
1 Market St                                 0
                                           ..
York St / Wellington St W (1)               0
York St / Wellington St W (2)               0
York University (Glendon Campus) - SMART    0
York University Station (North)             0
York University Station (South) - SMART     0
Name: count, Length: 1559, dtype: int64
2026-03-05 13:51:13 2026-03-13 15:41:43


In [179]:
import time

def geocode(query):
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "json", "limit": 1},
        headers={"User-Agent": "bike-sharing-analysis/1.0"},
        timeout=15
    )
    results = resp.json()
    return results[0] if results else None

for q in ["St. Andrew Subway Station, Toronto, Ontario", "NycLab, Toronto, Ontario"]:
    r = geocode(q)
    print(q, "->", r)
    time.sleep(1)


St. Andrew Subway Station, Toronto, Ontario -> None
NycLab, Toronto, Ontario -> None


In [180]:
for q in ["St Andrew subway station Toronto", "St Andrew Station Toronto TTC", "NycLab Toronto"]:
    r = geocode(q)
    print(q, "->", r)
    time.sleep(1)


St Andrew subway station Toronto -> None
St Andrew Station Toronto TTC -> None
NycLab Toronto -> None


In [181]:
resp = requests.get(
    "https://nominatim.openstreetmap.org/search",
    params={"q": "St Andrew Station Toronto", "format": "json", "limit": 1},
    headers={"User-Agent": "bike-sharing-analysis/1.0"},
    timeout=15
)
print(resp.status_code)
print(resp.text[:500])


200
[{"place_id":351784775,"licence":"Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright","osm_type":"node","osm_id":34710073,"lat":"43.6477917","lon":"-79.3848711","class":"railway","type":"station","place_rank":30,"importance":0.40580843572923003,"addresstype":"railway","name":"St. Andrew","display_name":"St. Andrew, St. Andrew Station Concourse, Financial District, Spadina—Fort York, Toronto, Ontario, M5H 4H2, Canada","boundingbox":["43.6427917","43.6527917","-79.3898711","-79.


In [182]:
r = geocode("St Andrew Station Toronto")
print("St Andrew:", r)
time.sleep(1)
r2 = geocode("NycLab")
print("NycLab (no city):", r2)


St Andrew: {'place_id': 351784775, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 34710073, 'lat': '43.6477917', 'lon': '-79.3848711', 'class': 'railway', 'type': 'station', 'place_rank': 30, 'importance': 0.40580843572923003, 'addresstype': 'railway', 'name': 'St. Andrew', 'display_name': 'St. Andrew, St. Andrew Station Concourse, Financial District, Spadina—Fort York, Toronto, Ontario, M5H 4H2, Canada', 'boundingbox': ['43.6427917', '43.6527917', '-79.3898711', '-79.3798711']}
NycLab (no city): None


In [183]:
from shapely.geometry import Point

pt = Point(-79.3848711, 43.6477917)
match = neigh[neigh.geometry.contains(pt)]
st_andrew_neigh = match['AREA_NAME'].iloc[0] if len(match) else None
print("St. Andrew Subway Station neighbourhood:", st_andrew_neigh)

df.loc[df['start_station_id'] == 8233.0, 'start_neighbourhood'] = st_andrew_neigh
print("remaining unmatched in 2026:", df.loc[(df['start_time'].dt.year == 2026) & df['start_neighbourhood'].isna()].shape[0])


St. Andrew Subway Station neighbourhood: Yonge-Bay Corridor
remaining unmatched in 2026: 4


In [184]:
mask_2025 = (df['start_time'].dt.year == 2025) & df['start_neighbourhood'].isna()
sub2025 = df.loc[mask_2025]
print(sub2025['start_station_id'].value_counts())
print(sub2025[['start_station_id','start_station_name']].drop_duplicates())


start_station_id
8067.0    291
8072.0    246
8132.0      7
Name: count, dtype: int64
          start_station_id         start_station_name
9632314             8067.0             Rogers Stadium
11640073            8072.0         Wellesley Station 
13697027            8132.0  Eglinton Ave E / Swift Dr
13827167               NaN                        NaN


In [185]:
nan_id_mask = mask_2025 & df['start_station_id'].isna()
print("rows with missing start_station_id entirely:", nan_id_mask.sum())

for q in ["Rogers Stadium, Toronto", "Wellesley Station, Toronto", "Eglinton Ave E / Swift Dr, Toronto"]:
    r = geocode(q)
    print(q, "->", r)
    time.sleep(1)


rows with missing start_station_id entirely: 7
Rogers Stadium, Toronto -> {'place_id': 351493126, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 1377018861, 'lat': '43.7461656', 'lon': '-79.4692345', 'class': 'leisure', 'type': 'stadium', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'leisure', 'name': 'Rogers Stadium', 'display_name': 'Rogers Stadium, Carl Hall Road, Downsview, York Centre, North York, Toronto, Ontario, M3K 2C9, Canada', 'boundingbox': ['43.7436731', '43.7483586', '-79.4732530', '-79.4651460']}
Wellesley Station, Toronto -> {'place_id': 351411412, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 387461259, 'lat': '43.6653371', 'lon': '-79.3839088', 'class': 'highway', 'type': 'bus_stop', 'place_rank': 30, 'importance': 0.40363877862033115, 'addresstype': 'highway', 'name': 'Wellesley Station', 'display_name': 'Welle

In [115]:
for q in ["Eglinton Avenue East and Swift Drive, Toronto", "Swift Drive, Toronto", "Eglinton Ave E, Toronto"]:
    r = geocode(q)
    print(q, "->", r)
    time.sleep(1)


Eglinton Avenue East and Swift Drive, Toronto -> None
Swift Drive, Toronto -> {'place_id': 351464875, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 234365693, 'lat': '43.7270522', 'lon': '-79.3186007', 'class': 'highway', 'type': 'residential', 'place_rank': 26, 'importance': 0.05341463569085218, 'addresstype': 'residential', 'name': 'Swift Drive', 'display_name': 'Swift Drive, Victoria Village, Don Valley East, North York, Toronto, Ontario, M4A 1K2, Canada', 'boundingbox': ['43.7267451', '43.7273593', '-79.3187185', '-79.3184829']}
Eglinton Ave E, Toronto -> {'place_id': 351376974, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 283010830, 'lat': '43.7201395', 'lon': '-79.3398128', 'class': 'highway', 'type': 'primary', 'place_rank': 26, 'importance': 0.05341463569085218, 'addresstype': 'primary', 'name': 'Eglinton Avenue East', 'display_name': 'Eglinton 

In [ ]:
fixes = {
    8067.0: Point(-79.4692345, 43.7461656), # rogers stadium
    8072.0: Point(-79.3839088, 43.6653371), # Wellesley Station
    8132.0: Point(-79.3186007, 43.7270522), # Eglinton Ave E / Swift Dr 
}

for station_id, pt in fixes.items():
    match = neigh[neigh.geometry.contains(pt)]
    name = match['AREA_NAME'].iloc[0] if len(match) else None
    df.loc[df['start_station_id'] == station_id, 'start_neighbourhood'] = name
    print(f"{station_id}: {name}")

remaining_2025 = df.loc[(df['start_time'].dt.year == 2025) & df['start_neighbourhood'].isna()]
remaining_2025.shape

8067.0 -> Downsview
8072.0 -> Church-Wellesley
8132.0 -> Victoria Village
remaining unmatched in 2025: 7 (of which missing start_station_id entirely: 7 )


In [189]:
# for yr in [2024, 2022]:
#     mask = (df['start_time'].dt.year == yr) & df['start_neighbourhood'].isna()
#     sub = df.loc[mask]



In [190]:
for sid in [7714.0, 7722.0]:
    row = all_stations[all_stations['station_id'] == sid]
    print(sid, "in all_stations:", not row.empty)
    if not row.empty:
        print(row)
    # check names from end_station columns in df for this id
    names_as_end = df.loc[df['end_station_id'] == sid, 'end_station_name'].dropna().unique()
    print("  names seen as end_station:", names_as_end[:5])


7714.0 in all_stations: False
  names seen as end_station: [], Categories (1561, str): [' George St S / The Esplanade', ' Grafton Ave / Roncesvalles Ave', ' Silverstone Dr / Stevenson Rd ', '1 Market St', ..., 'York St / Wellington St W (2)', 'York University (Glendon Campus) - SMART', 'York University Station (North)', 'York University Station (South) - SMART']
7722.0 in all_stations: False
  names seen as end_station: [], Categories (1561, str): [' George St S / The Esplanade', ' Grafton Ave / Roncesvalles Ave', ' Silverstone Dr / Stevenson Rd ', '1 Market St', ..., 'York St / Wellington St W (2)', 'York University (Glendon Campus) - SMART', 'York University Station (North)', 'York University Station (South) - SMART']


In [191]:
cols = ['trip_id','trip_duration','start_station_id','start_station_name','start_time',
        'end_station_id','end_station_name','end_time','bike_id','user_type']
print(df.loc[(df['start_time'].dt.year==2022) & (df['start_station_id']==7714.0), cols].head(3))
print(df.loc[(df['start_time'].dt.year==2022) & (df['start_station_id']==7722.0), cols].head(3))


           trip_id  trip_duration  start_station_id start_station_name  \
29801443  17755510          401.0            7714.0                NaN   
29802393  17756512         1055.0            7714.0                NaN   
29802790  17756930          818.0            7714.0                NaN   

                  start_time  end_station_id  \
29801443 2022-08-10 17:18:00          7426.0   
29802393 2022-08-10 17:40:00          7440.0   
29802790 2022-08-10 17:50:00          7175.0   

                            end_station_name            end_time  bike_id  \
29801443  Fleet St / Garrison Rd (Fort York) 2022-08-10 17:25:00   4839.0   
29802393   Martin Goodman Trail / Ontario Dr 2022-08-10 17:58:00    801.0   
29802790            HTO Park (Queens Quay W) 2022-08-10 18:03:00    611.0   

              user_type  
29801443  Casual Member  
29802393  Casual Member  
29802790  Casual Member  
           trip_id  trip_duration  start_station_id start_station_name  \
30101361  18095794     

In [192]:
r = geocode("2 Church St, Toronto")
print(r)


{'place_id': 352580988, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 3307844397, 'lat': '43.6479122', 'lon': '-79.3734044', 'class': 'place', 'type': 'house', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'place', 'name': '', 'display_name': '9 1/2, Church Street, St Lawrence-East Bayfront-The Islands, Toronto Centre, Toronto, Ontario, M5E 1M2, Canada', 'boundingbox': ['43.6478622', '43.6479622', '-79.3734544', '-79.3733544']}


In [193]:
pt = Point(-79.3730960, 43.6463914)
match = neigh[neigh.geometry.contains(pt)]
name = match['AREA_NAME'].iloc[0] if len(match) else None
df.loc[df['start_station_id'] == 7793.0, 'start_neighbourhood'] = name
print("2 Church St ->", name)
print("2024 remaining unmatched:", df.loc[(df['start_time'].dt.year==2024) & df['start_neighbourhood'].isna()].shape[0])
print("2022 remaining unmatched:", df.loc[(df['start_time'].dt.year==2022) & df['start_neighbourhood'].isna()].shape[0])


2 Church St -> St Lawrence-East Bayfront-The Islands
2024 remaining unmatched: 0
2022 remaining unmatched: 626


In [195]:
mask_2021 = (df['start_time'].dt.year == 2021) & df['start_neighbourhood'].isna()
sub2021 = df.loc[mask_2021]

print(sub2021[['start_station_id','start_station_name']].drop_duplicates())
print("missing start_station_id entirely:", sub2021['start_station_id'].isna().sum())


          start_station_id           start_station_name
31913890            7573.0  Torrens Ave / Broadview Ave
missing start_station_id entirely: 0


In [196]:
r = geocode("Torrens Ave / Broadview Ave, Toronto")
print(r)


{'place_id': 349211471, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 2893787038, 'lat': '43.6901783', 'lon': '-79.3550294', 'class': 'highway', 'type': 'bus_stop', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'highway', 'name': 'Torrens Avenue', 'display_name': 'Torrens Avenue, Broadview Avenue, Broadview North, Toronto—Danforth, East York, Toronto, Ontario, M4K 2T3, Canada', 'boundingbox': ['43.6901283', '43.6902283', '-79.3550794', '-79.3549794']}


In [197]:
pt = Point(-79.3550294, 43.6901783)
match = neigh[neigh.geometry.contains(pt)]
name = match['AREA_NAME'].iloc[0] if len(match) else None
df.loc[df['start_station_id'] == 7573.0, 'start_neighbourhood'] = name
print("Torrens Ave / Broadview Ave ->", name)
print("2021 remaining unmatched:", df.loc[(df['start_time'].dt.year==2021) & df['start_neighbourhood'].isna()].shape[0])


Torrens Ave / Broadview Ave -> Broadview North
2021 remaining unmatched: 0


In [199]:
for yr in [2020, 2019]:
    mask = (df['start_time'].dt.year == yr) & df['start_neighbourhood'].isna()
    sub = df.loc[mask]
    print(sub[['start_station_id','start_station_name']].drop_duplicates())
    print("missing start_station_id entirely:", sub['start_station_id'].isna().sum())
    print()


          start_station_id         start_station_name
36154995            7532.0  135 Queen’s Wharf - SMART
37846059            7394.0          Toronto Bike Shop
missing start_station_id entirely: 0

          start_station_id         start_station_name
38530378            7218.0      Make Invisible - 7218
38765582            7394.0          Toronto Bike Shop
39271849            7219.0            Wolfpack - 7219
39448133            7511.0                   PBSC-OPS
39985180            7393.0  55 Magnificent Rd. Garage
missing start_station_id entirely: 0



In [201]:
mask_2020 = (df['start_time'].dt.year == 2020) & df['start_neighbourhood'].isna()
sub2020 = df.loc[mask_2020]
print(sub2020[['start_station_id','start_station_name']].drop_duplicates())
print("missing start_station_id entirely:", sub2020['start_station_id'].isna().sum())


          start_station_id         start_station_name
36154995            7532.0  135 Queen’s Wharf - SMART
37846059            7394.0          Toronto Bike Shop
missing start_station_id entirely: 0


In [202]:
mask_2019 = (df['start_time'].dt.year == 2019) & df['start_neighbourhood'].isna()
sub2019 = df.loc[mask_2019]
print(sub2019[['start_station_id','start_station_name']].drop_duplicates())
print("missing start_station_id entirely:", sub2019['start_station_id'].isna().sum())


          start_station_id         start_station_name
38530378            7218.0      Make Invisible - 7218
38765582            7394.0          Toronto Bike Shop
39271849            7219.0            Wolfpack - 7219
39448133            7511.0                   PBSC-OPS
39985180            7393.0  55 Magnificent Rd. Garage
missing start_station_id entirely: 0


In [203]:
r_queens_wharf = geocode("135 Queens Wharf Rd, Toronto")
print("135 Queen's Wharf:", r_queens_wharf)


135 Queen's Wharf: {'place_id': 418220990, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 13854670808, 'lat': '43.6399700', 'lon': '-79.3993600', 'class': 'place', 'type': 'house', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'place', 'name': '', 'display_name': '135, Queens Wharf Road, Harbourfront-CityPlace, Downtown, Toronto, Ontario, M5V 4A7, Canada', 'boundingbox': ['43.6399200', '43.6400200', '-79.3994100', '-79.3993100']}


In [204]:
r_bike_shop = geocode("Toronto Bike Shop")
print("Toronto Bike Shop:", r_bike_shop)


Toronto Bike Shop: {'place_id': 352846101, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 13524485391, 'lat': '43.6745189', 'lon': '-79.4318713', 'class': 'shop', 'type': 'bicycle', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'shop', 'name': 'Road Off Road Bike Shop', 'display_name': 'Road Off Road Bike Shop, 1157, Davenport Road, Wychwood, Davenport, Toronto, Ontario, M6H 2G7, Canada', 'boundingbox': ['43.6744689', '43.6745689', '-79.4319213', '-79.4318213']}


In [205]:
r_bike_shop2 = geocode('"Toronto Bike Shop"')
print("exact phrase:", r_bike_shop2)


exact phrase: {'place_id': 351532976, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 13715060101, 'lat': '43.6713997', 'lon': '-79.4219238', 'class': 'shop', 'type': 'bicycle', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'shop', 'name': '11 Bike Shop', 'display_name': '11 Bike Shop, 677, Dupont Street, Dovercourt Village, University—Rosedale, Toronto, Ontario, M6G 1Z5, Canada', 'boundingbox': ['43.6713497', '43.6714497', '-79.4219738', '-79.4218738']}


In [206]:
r_magnificent = geocode("55 Magnificent Rd, Toronto")
print("55 Magnificent Rd:", r_magnificent)


55 Magnificent Rd: {'place_id': 419219631, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 13882636235, 'lat': '43.6128200', 'lon': '-79.5117100', 'class': 'place', 'type': 'house', 'place_rank': 30, 'importance': 8.130235751883753e-05, 'addresstype': 'place', 'name': '', 'display_name': '55, Magnificent Road, Mimico-Queensway, Etobicoke—Lakeshore, Etobicoke, Toronto, Ontario, M2Z 5W8, Canada', 'boundingbox': ['43.6127700', '43.6128700', '-79.5117600', '-79.5116600']}


In [207]:
r_invisible = geocode("Make Invisible, Toronto")
print("Make Invisible:", r_invisible)


Make Invisible: None


In [208]:
r_wolfpack = geocode("Wolfpack, Toronto")
print("Wolfpack:", r_wolfpack)


Wolfpack: None


In [209]:
r_pbsc = geocode("PBSC, Toronto")
print("PBSC:", r_pbsc)


PBSC: None


In [210]:
pt_wharf = Point(-79.3993600, 43.6399700)
match_wharf = neigh[neigh.geometry.contains(pt_wharf)]
name_wharf = match_wharf['AREA_NAME'].iloc[0] if len(match_wharf) else None
df.loc[df['start_station_id'] == 7532.0, 'start_neighbourhood'] = name_wharf
print("135 Queen's Wharf ->", name_wharf)


135 Queen's Wharf -> Harbourfront-CityPlace


In [211]:
pt_magnificent = Point(-79.5117100, 43.6128200)
match_magnificent = neigh[neigh.geometry.contains(pt_magnificent)]
name_magnificent = match_magnificent['AREA_NAME'].iloc[0] if len(match_magnificent) else None
df.loc[df['start_station_id'] == 7393.0, 'start_neighbourhood'] = name_magnificent
print("55 Magnificent Rd ->", name_magnificent)


55 Magnificent Rd -> Mimico-Queensway


In [212]:
print("2020 remaining unmatched:", df.loc[(df['start_time'].dt.year==2020) & df['start_neighbourhood'].isna()].shape[0])
print("2019 remaining unmatched:", df.loc[(df['start_time'].dt.year==2019) & df['start_neighbourhood'].isna()].shape[0])


2020 remaining unmatched: 17
2019 remaining unmatched: 74


In [214]:
mask_2018 = (df['start_time'].dt.year == 2018) & df['start_neighbourhood'].isna()
sub2018 = df.loc[mask_2018]
print(sub2018[['start_station_id','start_station_name']].drop_duplicates())
print("missing start_station_id entirely:", sub2018['start_station_id'].isna().sum())


          start_station_id start_station_name
42608879            7219.0    Wolfpack - 7219
missing start_station_id entirely: 0


In [215]:
mask_2017 = (df['start_time'].dt.year == 2017) & df['start_neighbourhood'].isna()
sub2017 = df.loc[mask_2017]
unique_2017 = sub2017[['start_station_id','start_station_name']].drop_duplicates()
print("unique unmatched stations:", len(unique_2017))
print(unique_2017.head(20))
print("missing start_station_id entirely:", sub2017['start_station_id'].isna().sum())


unique unmatched stations: 271
          start_station_id                       start_station_name
43072804               NaN              Princess St / Adelaide St E
43072805               NaN            Fort York  Blvd / Capreol Crt
43072807               NaN  Elizabeth St / Edward St (Bus Terminal)
43072809               NaN                   Bay St / St. Joseph St
43072810               NaN          Bay St / College St (East Side)
43072811               NaN                   Bremner Blvd / Rees St
43072813               NaN                  University Ave / Elm St
43072814               NaN           King St W / Bay St (East Side)
43072815               NaN                Widmer St / Adelaide St W
43072816               NaN             Fort York Blvd / Garrison Rd
43072818               NaN       Michael Sweet Ave / St. Patrick St
43072819               NaN                  Jarvis St / Isabella St
43072820               NaN               University Ave / King St W
43072821         

In [216]:
names_2017 = set(unique_2017['start_station_name'].dropna().unique())
known_names = set(all_stations['name'].dropna().unique())
exact_matches = names_2017 & known_names
print(f"{len(exact_matches)} / {len(names_2017)} 2017 station names match exactly against our known station list")
print(list(names_2017 - known_names)[:15])


193 / 271 2017 station names match exactly against our known station list
["Wellesley St / Queen's Park Cres", 'Beverly  St / Dundas St W', '424 Wellington St W', 'Bloor St W / Shaw Ave - SMART', 'Lake Shore Blvd W / Ontario Dr(Ontario Place)', 'Queen St W / York St (City Hall)', 'Summerhill Station', 'King / Cowan Ave - SMART', 'Queen St W / Portland St', 'Cariboo St / Rail Path', 'Queen St E / Larchmount Ave', 'Dufferin St / Bloor St', 'Simcoe St / Wellington St W', 'Boston Ave / Queen St E', 'Castle Frank Station']


In [217]:
name_to_neigh = all_with_neigh.dropna(subset=['name']).drop_duplicates(subset='name', keep='last').set_index('name')['neighbourhood']

still_missing = df['start_neighbourhood'].isna()
fallback = df.loc[still_missing, 'start_station_name'].map(name_to_neigh)
df.loc[still_missing, 'start_neighbourhood'] = df.loc[still_missing, 'start_neighbourhood'].fillna(fallback)

print("2017 remaining unmatched:", df.loc[(df['start_time'].dt.year==2017) & df['start_neighbourhood'].isna()].shape[0], "/", (df['start_time'].dt.year==2017).sum())


2017 remaining unmatched: 271221 / 1492368


In [218]:
remaining_2017 = df.loc[(df['start_time'].dt.year==2017) & df['start_neighbourhood'].isna()]
print(remaining_2017['start_station_name'].value_counts().head(30))


start_station_name
Simcoe St / Wellington St W                    14176
Queen St W / Portland St                       11221
Ontario Place Blvd / Remembrance Dr            10994
Michael Sweet Ave / St. Patrick St             10786
Beverly  St / Dundas St W                       9029
Fort York  Blvd / Capreol Crt                   8773
Dockside Dr / Queens Quay E (Sugar Beach)       8183
University Ave / King St W                      7577
Bay St / Bloor St W                             7166
Bathurst St / Queens Quay W                     6942
Beverly St / College St                         6646
Victoria St / Gould St (Ryerson University)     6364
Bremner Blvd / Spadina Ave                      6243
University Ave / College St                     6038
Wellesley St E / Yonge St (Green P)             5853
Queen St W / York St (City Hall)                5753
Bloor St / Brunswick Ave                        5514
Queens Park / Bloor St W                        5285
25 York St (ACC/Union Stati

In [219]:
mask_2016 = (df['start_time'].dt.year == 2016) & df['start_neighbourhood'].isna()
sub2016 = df.loc[mask_2016]
print(sub2016['start_station_name'].value_counts().head(20))


start_station_name
Widmer St / Adelaide St                      5710
Simcoe St / Wellington St W                  5646
Dundas St / Yonge St                         5465
Queen St W / Portland St                     4655
Princess St / Adelaide St                    4182
HTO Park (Queen's Quay W)                    3986
Sherbourne St / Wellesley St                 3945
Fort York  Blvd / Capreol Crt                3873
Beverly  St / Dundas St W                    3736
Bay St / Bloor St W                          3715
Dockside Dr / Queens Quay E (Sugar Beach)    3707
University Ave / King St W                   3592
Temperance St / Yonge St                     3468
College St W / Huron St                      3331
Bremner Blvd / Spadina Ave                   3292
Queens Park / Bloor St W                     3190
424 Wellington St. W                         3146
College St W / Major St                      3046
Wellesley St E / Yonge St Green P            2799
St George St / Bloor St W      

In [220]:
from difflib import get_close_matches

known_names_list = list(known_names)
still_missing = df['start_neighbourhood'].isna()
unmatched_names = df.loc[still_missing, 'start_station_name'].dropna().unique()
print(f"{len(unmatched_names)} unique unmatched names to fuzzy-match")

fuzzy_map = {}
for name in unmatched_names:
    hits = get_close_matches(name, known_names_list, n=1, cutoff=0.82)
    if hits:
        fuzzy_map[name] = hits[0]

print(f"{len(fuzzy_map)} / {len(unmatched_names)} names matched")
list(fuzzy_map.items())[:15]


105 unique unmatched names to fuzzy-match
63 / 105 names matched


[('Fort York  Blvd / Capreol Crt', 'Fort York  Blvd / Capreol Ct'),
 ('University Ave / King St W', 'University Ave / Queen St W'),
 ('Beverly St / College St', 'Beverley St / College St'),
 ('Queens Park / Bloor St W', "Queen's Park / Bloor St W"),
 ('25 York St (ACC/Union Station South)', '25 York St – Union Station South'),
 ('Simcoe St / Wellington St W', 'Simcoe St / Wellington St W South'),
 ('Queen St W / Portland St', 'King St W / Portland St '),
 ('Queen St W / Gladstone Ave', 'Queen St W / Close Ave'),
 ('Wellington St W / Stafford St', 'King St W / Stafford St'),
 ('Bridgeman Ave / Kendal Ave', 'Bridgman Ave / Kendal Ave'),
 ('Bloor St / Brunswick Ave', 'Wells St / Brunswick Ave'),
 ('Seaton St / Dundas St E', 'Seaton St / Dundas St E - SMART'),
 ('University Ave / Gerrard St W', 'University Ave / Pearl St '),
 ('Sumach St / Carlton St. (Riverdale Farm)',
  'Sumach St / Carlton St (Riverdale Farm)'),
 ('Beverly  St / Dundas St W', 'Beverley St / Dundas St W')]

In [221]:
import re
from difflib import SequenceMatcher

def tokenize(s):
    return re.findall(r"[A-Za-z0-9']+", s.lower())

def is_safe_match(orig, candidate):
    set1, set2 = set(tokenize(orig)), set(tokenize(candidate))
    unmatched1, unmatched2 = list(set1 - set2), list(set2 - set1)
    if not unmatched1 or not unmatched2:
        return True
    for w in unmatched1:
        best = max((SequenceMatcher(None, w, w2).ratio() for w2 in unmatched2), default=0)
        if best < 0.6:
            return False
    return True

safe_fuzzy_map = {k: v for k, v in fuzzy_map.items() if is_safe_match(k, v)}
rejected = {k: v for k, v in fuzzy_map.items() if k not in safe_fuzzy_map}
print(f"{len(safe_fuzzy_map)} safe matches, {len(rejected)} rejected")
print("--- rejected (different street substituted) ---")
for k, v in rejected.items():
    print(k, "->", v)


54 safe matches, 9 rejected
--- rejected (different street substituted) ---
University Ave / King St W -> University Ave / Queen St W
Queen St W / Portland St -> King St W / Portland St 
Queen St W / Gladstone Ave -> Queen St W / Close Ave
Wellington St W / Stafford St -> King St W / Stafford St
Bloor St / Brunswick Ave -> Wells St / Brunswick Ave
University Ave / Gerrard St W -> University Ave / Pearl St 
York St / King St W - SMART -> Niagara St / King St W - SMART
Beverly St / College St W -> Beverley St / College St
Temperance St / Yonge St -> Temperance St / Bay St


In [222]:
fuzzy_name_to_neigh = {orig: name_to_neigh.get(matched) for orig, matched in safe_fuzzy_map.items()}

still_missing = df['start_neighbourhood'].isna()
fallback2 = df.loc[still_missing, 'start_station_name'].map(fuzzy_name_to_neigh)
df.loc[still_missing, 'start_neighbourhood'] = df.loc[still_missing, 'start_neighbourhood'].fillna(fallback2)

print("2016 remaining unmatched:", df.loc[(df['start_time'].dt.year==2016) & df['start_neighbourhood'].isna()].shape[0], "/", (df['start_time'].dt.year==2016).sum())
print("2017 remaining unmatched:", df.loc[(df['start_time'].dt.year==2017) & df['start_neighbourhood'].isna()].shape[0], "/", (df['start_time'].dt.year==2017).sum())


2016 remaining unmatched: 49028 / 367961
2017 remaining unmatched: 144262 / 1492368


In [223]:
remaining_now = df.loc[df['start_neighbourhood'].isna() & df['start_time'].dt.year.isin([2016,2017])]
print(remaining_now['start_station_name'].value_counts().head(20))


start_station_name
Queen St W / Portland St                         15876
Ontario Place Blvd / Remembrance Dr              13314
Dockside Dr / Queens Quay E (Sugar Beach)        11890
University Ave / King St W                       11169
Bay St / Bloor St W                              10881
Michael Sweet Ave / St. Patrick St               10786
Bremner Blvd / Spadina Ave                        9535
Victoria St / Gould St (Ryerson University)       8213
Bloor St / Brunswick Ave                          7963
Queen St W / York St (City Hall)                  7821
Yonge St / Harbour St                             7572
Bathurst St / Queens Quay W                       7298
University Ave / Gerrard St W                     6874
Wellesley St E / Yonge St (Green P)               5853
Cherry St / Mill St                               5437
Queen St W / Gladstone Ave                        5128
St. George St / Russell St - SMART                3810
Temperance St / Yonge St                      

In [225]:
df['year'] = df.start_time.dt.year

In [66]:
df.columns

Index(['trip_id', 'trip_duration', 'start_station_id', 'start_time',
       'start_station_name', 'end_station_id', 'end_time', 'end_station_name',
       'bike_id', 'user_type', 'bike_model', 'start_neighbourhood', 'year'],
      dtype='str')

In [226]:
(df.loc[df['start_station_id'].isna() & ~df['start_station_name'].isna()]
    .groupby('year', as_index=False)
    .agg(unique_missings = ('start_station_name', 'nunique'))
    )

,year,unique_missings
0,2016,200
1,2017,271


In [ ]:
# 293 stations with names but without ids, 1394851 trips. 

In [227]:
df.memory_usage(deep=True).sum()/1e9

np.float64(3.947918506)

In [228]:
df.info()

<class 'pandas.DataFrame'>
Index: 44599780 entries, 0 to 44599780
Data columns (total 13 columns):
 #   Column               Dtype         
---  ------               -----         
 0   trip_id              int64         
 1   trip_duration        float64       
 2   start_station_id     float32       
 3   start_time           datetime64[us]
 4   start_station_name   category      
 5   end_station_id       float32       
 6   end_time             datetime64[us]
 7   end_station_name     category      
 8   bike_id              float32       
 9   user_type            category      
 10  bike_model           category      
 11  start_neighbourhood  str           
 12  year                 int32         
dtypes: category(4), datetime64[us](2), float32(3), float64(1), int32(1), int64(1), str(1)
memory usage: 3.7 GB


In [229]:
df['start_neighbourhood']=df['start_neighbourhood'].astype('category')

In [230]:
df.memory_usage(deep=True).sum()/1e9

np.float64(2.854498218)

In [231]:
df.to_parquet('bikeshare_neigh.parquet', index=False)

# Neighbourhoods

In [165]:
# read neighbourhoods from the files
neigh = gpd.read_file('toronto-neighbourhoods.geojson')[['AREA_NAME', 'geometry']]

In [167]:
# connect stations with neighbourhoods
stations_with_neigh = gpd.sjoin(stations_gdf, neigh, how='left', predicate='within')
stations_with_neigh = stations_with_neigh[['station_id', 'name', 'AREA_NAME']].rename(columns={'AREA_NAME': 'neighbourhood'})

print("stations total:", len(stations_with_neigh))
print("unmatched with neigh:", stations_with_neigh['neighbourhood'].isna().sum())
stations_with_neigh.head()


stations total: 1070
unmatched with neigh: 0


,station_id,name,neighbourhood
0,7000,Fort York Blvd / Capreol Ct,Harbourfront-CityPlace
1,7001,Wellesley Station Green P,Church-Wellesley
2,7002,St. George St / Bloor St W,University
3,7003,Madison Ave / Bloor St W,Annex
4,7004,University Ave / Elm St,Kensington-Chinatown


In [168]:
station_to_neigh = stations_with_neigh.set_index('station_id')['neighbourhood']
df['start_neighbourhood'] = df['start_station_id'].map(station_to_neigh)

print("trips with no neighbourhood", df['start_neighbourhood'].isna().sum(), "/", len(df))


trips with no neighbourhood 4590165 / 44599780


# Wards